In [1]:
import joblib
import numpy as np
import os

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

In [2]:
save_dir = "data/processed"

raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
temperature = np.load(os.path.join(save_dir, "temperature.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
scaler      = joblib.load(os.path.join(save_dir, "scaler.pkl"))

num_train_samples = len(train_idx)
num_val_samples   = len(val_idx)
num_test_samples  = len(test_idx)

temp_mean = scaler.mean_[1]
temp_std  = scaler.scale_[1]

print(f"raw_data:    {raw_data.shape}")
print(f"temperature: {temperature.shape}")
print(f"train/val/test: {num_train_samples} / {num_val_samples} / {num_test_samples}")

temperature_normalized = (temperature - temp_mean) / temp_std

raw_data:    (420451, 14)
temperature: (420451,)
train/val/test: 210225 / 105112 / 105114


In [3]:
sampling_rate   = 6
sequence_length = 120
delay           = sampling_rate * (sequence_length + 24 - 1)
batch_size      = 256

class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Valid starting indices: each sequence of length sequence_length
        # sampled every sampling_rate steps needs:
        # (sequence_length - 1) * sampling_rate + 1 rows ahead
        self.indices = np.arange(start_index, end_index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Sample every `sampling_rate` steps for `sequence_length` steps
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # Target is `delay` steps ahead of the sequence start
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [4]:
train_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train_samples,
)

val_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train_samples, end_index=num_train_samples + num_val_samples,
)

test_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train_samples + num_val_samples, end_index=len(raw_data) - delay,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# Quick sanity check
for inputs, targets in train_loader:
    print("Input shape:", inputs.shape)   # (batch_size, sequence_length, num_features)
    print("Target shape:", targets.shape) # (batch_size,)
    break

Input shape: torch.Size([256, 120, 14])
Target shape: torch.Size([256])


In [5]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    total_mae  = 0.0
    n          = 0
    with torch.set_grad_enabled(training):
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            preds = model(inputs)
            loss  = criterion(preds, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            total_mae  += torch.sum(torch.abs(preds - targets)).item()
            n          += inputs.size(0)
   # mae_celsius = (total_mae / n) * temp_std
    return total_loss / n, total_mae / n   # MAE already in °C


def get_predictions(model, dataset):
    model.eval()
    all_preds     = []
    all_targets   = []
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    with torch.no_grad():
        for inputs, targets in loader:
            preds = model(inputs.to(device)).cpu().numpy()
            all_preds.append(preds * temp_std + temp_mean)  # un-normalize predictions
            all_targets.append(targets.numpy())              # targets already in °C
    return np.concatenate(all_preds), np.concatenate(all_targets)

In [17]:
class ModelCheckpoint:
    """Saves the best model based on a monitored metric."""
    def __init__(self, filepath, monitor="val_mae", mode="min", verbose=True):
        self.filepath = filepath
        self.monitor  = monitor
        self.verbose  = verbose
        self.best     = float("inf") if mode == "min" else float("-inf")
        self.mode     = mode

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = value < self.best if self.mode == "min" else value > self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f"  ✓ Best model saved ({self.monitor}: {value:.2f}°C)")
        return improved


class EarlyStopping:
    """Stops training when a monitored metric stops improving."""
    def __init__(self, monitor="val_mae", patience=5, min_delta=1e-4, mode="min"):
        self.monitor     = monitor
        self.patience    = patience
        self.min_delta   = min_delta
        self.mode        = mode
        self.best        = float("inf") if mode == "min" else float("-inf")
        self.counter     = 0
        self.should_stop = False

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = (value < self.best - self.min_delta if self.mode == "min"
                    else value > self.best + self.min_delta)
        if improved:
            self.best    = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  Early stopping triggered (no improvement for {self.patience} epochs)")
        return improved


class ReduceLROnPlateau:
    """Wraps PyTorch scheduler with the same callback interface."""
    def __init__(self, optimizer, monitor="val_mae", patience=3, factor=0.5):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=patience, factor=factor
        )
        self.monitor = monitor

    def step(self, metrics, model=None):
        self.scheduler.step(metrics[self.monitor])

In [7]:
class LSTMModel(nn.Module):
    def __init__(self, num_features, hidden_size=16):
        super().__init__()
        self.lstm   = nn.LSTM(input_size=num_features, hidden_size=hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # out: (batch, seq_len, hidden_size)
        # we only need the last timestep → out[:, -1, :]
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :]).squeeze(-1)  # (batch,)

In [8]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = LSTMModel(num_features=raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/jena_lstm")

callbacks = [
    ModelCheckpoint("jena_lstm_best.pt", monitor="val_mae"),
]

# --- Training loop ---
epochs = 10
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae:.2f}°C | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f}°C")

    for cb in callbacks:
        cb.step(metrics, model) if isinstance(cb, ModelCheckpoint) else cb.step(metrics)

writer.close()

Epoch 01 — train loss: 62.2969, train MAE: 5.89°C | val loss: 23.3830, val MAE: 3.57°C
  ✓ Best model saved (val_mae: 3.57°C)
Epoch 02 — train loss: 17.3974, train MAE: 3.12°C | val loss: 12.5466, val MAE: 2.68°C
  ✓ Best model saved (val_mae: 2.68°C)
Epoch 03 — train loss: 12.0960, train MAE: 2.68°C | val loss: 10.6456, val MAE: 2.52°C
  ✓ Best model saved (val_mae: 2.52°C)
Epoch 04 — train loss: 10.4906, train MAE: 2.52°C | val loss: 10.0479, val MAE: 2.46°C
  ✓ Best model saved (val_mae: 2.46°C)
Epoch 05 — train loss: 9.8909, train MAE: 2.45°C | val loss: 9.9521, val MAE: 2.46°C
Epoch 06 — train loss: 9.6027, train MAE: 2.42°C | val loss: 9.6695, val MAE: 2.43°C
  ✓ Best model saved (val_mae: 2.43°C)
Epoch 07 — train loss: 9.2197, train MAE: 2.37°C | val loss: 9.6517, val MAE: 2.42°C
  ✓ Best model saved (val_mae: 2.42°C)
Epoch 08 — train loss: 9.1690, train MAE: 2.36°C | val loss: 9.2351, val MAE: 2.37°C
  ✓ Best model saved (val_mae: 2.37°C)
Epoch 09 — train loss: 8.9265, train MA

In [9]:
# --- Reload best and evaluate ---
model.load_state_dict(torch.load("jena_lstm_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae:.2f}°C")


Test MAE: 2.57°C


In [10]:
class StackedLSTMModel(nn.Module):
    def __init__(self, num_features, hidden_size=16):
        super().__init__()
        self.lstm   = nn.LSTM(input_size=num_features, hidden_size=hidden_size, num_layers = 2, batch_first=True)
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # out: (batch, seq_len, hidden_size)
        # we only need the last timestep → out[:, -1, :]
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :]).squeeze(-1)  # (batch,)

In [11]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = StackedLSTMModel(num_features=raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/jena_stacked_lstm")

callbacks = [
    ModelCheckpoint("jena_stacked_lstm_best.pt", monitor="val_mae"),
]

# --- Training loop ---
epochs = 10
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae:.2f}°C | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f}°C")

    for cb in callbacks:
        cb.step(metrics, model) if isinstance(cb, ModelCheckpoint) else cb.step(metrics)

writer.close()

Epoch 01 — train loss: 58.1690, train MAE: 5.65°C | val loss: 22.8260, val MAE: 3.49°C
  ✓ Best model saved (val_mae: 3.49°C)
Epoch 02 — train loss: 16.1086, train MAE: 2.99°C | val loss: 12.3673, val MAE: 2.63°C
  ✓ Best model saved (val_mae: 2.63°C)
Epoch 03 — train loss: 11.5043, train MAE: 2.61°C | val loss: 9.9625, val MAE: 2.40°C
  ✓ Best model saved (val_mae: 2.40°C)
Epoch 04 — train loss: 9.8797, train MAE: 2.44°C | val loss: 9.7005, val MAE: 2.39°C
  ✓ Best model saved (val_mae: 2.39°C)
Epoch 05 — train loss: 8.9832, train MAE: 2.33°C | val loss: 9.5685, val MAE: 2.39°C
  ✓ Best model saved (val_mae: 2.39°C)
Epoch 06 — train loss: 8.4397, train MAE: 2.26°C | val loss: 9.8235, val MAE: 2.44°C
Epoch 07 — train loss: 8.5014, train MAE: 2.27°C | val loss: 9.4820, val MAE: 2.39°C
  ✓ Best model saved (val_mae: 2.39°C)
Epoch 08 — train loss: 7.8345, train MAE: 2.18°C | val loss: 9.8026, val MAE: 2.43°C
Epoch 09 — train loss: 7.3615, train MAE: 2.11°C | val loss: 10.0720, val MAE: 2.

In [12]:
# --- Reload best and evaluate ---
model.load_state_dict(torch.load("jena_stacked_lstm_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae:.2f}°C")


Test MAE: 2.59°C


In [20]:
class BidirectionalLSTMModel(nn.Module):
    def __init__(self, num_features, hidden_size=16):
        super().__init__()
        self.rnn  = nn.LSTM(input_size=num_features, hidden_size=hidden_size,
                           batch_first=True, bidirectional=True)
        # bidirectional doubles the output size
        self.head = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :]).squeeze(-1)

In [21]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = BidirectionalLSTMModel(num_features=raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/jena_bidirectional_lstm")

callbacks = [
    ModelCheckpoint("jena_bidirectional_lstm_best.pt", monitor="val_mae"),
    EarlyStopping(monitor="val_mae", patience=7),
    ReduceLROnPlateau(optimizer, monitor="val_mae", patience=3, factor=0.5),
]

# --- Training loop ---
epochs = 50
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae:.2f}°C | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f}°C")

    for cb in callbacks:
        cb.step(metrics, model)

    if any(isinstance(cb, EarlyStopping) and cb.should_stop for cb in callbacks):
        print(f"  Stopped at epoch {epoch}")
        break

writer.close()

Epoch 01 — train loss: 127.9717, train MAE: 9.37°C | val loss: 95.7318, val MAE: 7.98°C
  ✓ Best model saved (val_mae: 7.98°C)
Epoch 02 — train loss: 74.0797, train MAE: 6.88°C | val loss: 58.3981, val MAE: 6.06°C
  ✓ Best model saved (val_mae: 6.06°C)
Epoch 03 — train loss: 48.9732, train MAE: 5.48°C | val loss: 38.5034, val MAE: 4.81°C
  ✓ Best model saved (val_mae: 4.81°C)
Epoch 04 — train loss: 34.3336, train MAE: 4.48°C | val loss: 26.7772, val MAE: 3.90°C
  ✓ Best model saved (val_mae: 3.90°C)
Epoch 05 — train loss: 25.5704, train MAE: 3.80°C | val loss: 20.1935, val MAE: 3.35°C
  ✓ Best model saved (val_mae: 3.35°C)
Epoch 06 — train loss: 20.5697, train MAE: 3.39°C | val loss: 16.6431, val MAE: 3.05°C
  ✓ Best model saved (val_mae: 3.05°C)
Epoch 07 — train loss: 17.7442, train MAE: 3.16°C | val loss: 14.6139, val MAE: 2.89°C
  ✓ Best model saved (val_mae: 2.89°C)
Epoch 08 — train loss: 15.8025, train MAE: 3.00°C | val loss: 13.1797, val MAE: 2.77°C
  ✓ Best model saved (val_mae:

KeyboardInterrupt: 

In [22]:
model.load_state_dict(torch.load("jena_bidirectional_lstm_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae:.2f}°C")


Test MAE: 2.53°C
